# Week 6 - Spark Architecture and Data Processing using Apache Spark

## Celebal Technologies Summer Internship 2026

### Objective

This notebook demonstrates the implementation of Apache Spark concepts including Spark Architecture, Lazy Evaluation, DataFrame Transformations, Schema Handling, CSV and Parquet processing, Predicate Pushdown concepts, and an end-to-end data processing pipeline using PySpark.

---

# Step 1 - Environment Setup

In this section, PySpark is installed and a Spark Session is created to initialize the Spark application.

In [ ]:
!pip install pyspark

## Create Spark Session

The Spark Session acts as the entry point for all Spark DataFrame operations.

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Celebal Week 6 Assignment") \
    .getOrCreate()

print("Spark Session Created Successfully!")

Spark Session Created Successfully!


In [ ]:
import os

print(os.listdir())

['.config', 'ecommerce_orders.csv', 'sample_data']


## Step 2 — Reading the Dataset

The e-commerce dataset is loaded from a CSV file using Spark. Schema inference is enabled to automatically detect the appropriate data types for each column.

In [ ]:
df = spark.read.csv(
    "ecommerce_orders.csv",
    header=True,
    inferSchema=True
)

print("Dataset Loaded Successfully!\n")

Dataset Loaded Successfully!



## Step 3 — Exploring the Dataset

The first few records are displayed to understand the dataset structure and verify that the data has been loaded correctly.

In [ ]:
df.show(5)

+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
|user_id|product_id|   category|old_name|  price|base_price|amount|   status|region|priority|
+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
|   U001|      P101|Electronics|  Item_1| 178.18|       151|534.54|Completed| South|    High|
|   U002|      P102|Electronics|  Item_2| 1752.3|      1485|8761.5|Completed|  West|    High|
|   U003|      P103|Electronics|  Item_3| 343.38|       291|686.76|Completed| North|     Low|
|   U004|      P104|   Clothing|  Item_4|1847.88|      1566|9239.4|  Pending| South|  Medium|
|   U005|      P105|     Sports|  Item_5| 789.42|       669|789.42|Completed|  West|  Medium|
+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
only showing top 5 rows


## Step 4 — Schema Inspection

Spark automatically infers the schema while reading the CSV file. The schema is displayed below to verify the detected data types.

In [ ]:
df.printSchema()

root
 |-- user_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- old_name: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)



## Step 5 — Filtering and Selecting Data

The dataset is filtered to retrieve only products belonging to the **Electronics** category. Only the required columns (`product_id` and `price`) are selected.

In [ ]:
from pyspark.sql.functions import col

electronics_df = df.filter(
    col("category") == "Electronics"
).select(
    "product_id",
    "price"
)

electronics_df.show()

+----------+-------+
|product_id|  price|
+----------+-------+
|      P101| 178.18|
|      P102| 1752.3|
|      P103| 343.38|
|      P110| 227.74|
|      P111|1036.04|
|      P113|1589.46|
|      P122|1047.84|
|      P137|2305.72|
|      P142| 264.32|
+----------+-------+



## Step 6 — Modifying the DataFrame

This step demonstrates DataFrame modification by:

- Renaming a column
- Casting the data type of an existing column
- Verifying the updated schema

In [ ]:
from pyspark.sql.types import StringType, DoubleType

df_string = df.withColumn(
    "price",
    col("price").cast(StringType())
)

updated_df = df_string.withColumnRenamed(
    "old_name",
    "new_name"
).withColumn(
    "price",
    col("price").cast(DoubleType())
)

updated_df.printSchema()
updated_df.show(5)

root
 |-- user_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- category: string (nullable = true)
 |-- new_name: string (nullable = true)
 |-- price: double (nullable = true)
 |-- base_price: integer (nullable = true)
 |-- amount: double (nullable = true)
 |-- status: string (nullable = true)
 |-- region: string (nullable = true)
 |-- priority: string (nullable = true)

+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
|user_id|product_id|   category|new_name|  price|base_price|amount|   status|region|priority|
+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
|   U001|      P101|Electronics|  Item_1| 178.18|       151|534.54|Completed| South|    High|
|   U002|      P102|Electronics|  Item_2| 1752.3|      1485|8761.5|Completed|  West|    High|
|   U003|      P103|Electronics|  Item_3| 343.38|       291|686.76|Completed| North|     Low|
|   U004|      P104|   Clothing|  I

## Step 7 — Applying Multiple Filter Conditions

The DataFrame is filtered to retrieve only orders where:

- Status is **Completed**
- Amount is greater than **1000**

In [ ]:
completed_orders = df.filter(
    (col("status") == "Completed") &
    (col("amount") > 1000)
)

completed_orders.show()

+-------+----------+-----------+--------+-------+----------+-------+---------+------+--------+
|user_id|product_id|   category|old_name|  price|base_price| amount|   status|region|priority|
+-------+----------+-----------+--------+-------+----------+-------+---------+------+--------+
|   U002|      P102|Electronics|  Item_2| 1752.3|      1485| 8761.5|Completed|  West|    High|
|   U015|      P115|    Grocery| Item_15|2154.68|      1826|2154.68|Completed| North|  Medium|
|   U017|      P117|  Furniture| Item_17| 1073.8|       910| 4295.2|Completed|  East|    High|
|   U020|      P120|   Clothing| Item_20|1348.74|      1143|5394.96|Completed| North|    High|
|   U029|      P129|    Grocery| Item_29| 1298.0|      1100| 1298.0|Completed|  East|  Medium|
|   U038|      P138|   Clothing| Item_38|  790.6|       670| 3162.4|Completed| South|     Low|
|   U039|      P139|     Sports| Item_39| 1510.4|      1280| 6041.6|Completed|  West|  Medium|
|   U045|      P145|  Furniture| Item_45|2069.72| 

## Step 8 — Creating a Calculated Column

A new column named **final_price** is created by applying an 18% tax to the **base_price** column.

In [ ]:
final_price_df = df.withColumn(
    "final_price",
    col("base_price") * 1.18
)

final_price_df.show(5)

+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+------------------+
|user_id|product_id|   category|old_name|  price|base_price|amount|   status|region|priority|       final_price|
+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+------------------+
|   U001|      P101|Electronics|  Item_1| 178.18|       151|534.54|Completed| South|    High|178.17999999999998|
|   U002|      P102|Electronics|  Item_2| 1752.3|      1485|8761.5|Completed|  West|    High|            1752.3|
|   U003|      P103|Electronics|  Item_3| 343.38|       291|686.76|Completed| North|     Low|            343.38|
|   U004|      P104|   Clothing|  Item_4|1847.88|      1566|9239.4|  Pending| South|  Medium|1847.8799999999999|
|   U005|      P105|     Sports|  Item_5| 789.42|       669|789.42|Completed|  West|  Medium|            789.42|
+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+---

## Step 9 — Writing Data in Parquet Format

The processed DataFrame is written in Parquet format. Parquet is a columnar storage format that provides better compression and faster analytical queries than CSV.

In [ ]:
df.write.mode("overwrite").parquet("input_parquet")

print("Parquet file created successfully!")

Parquet file created successfully!


## Step 10 — Reading the Parquet File

The Parquet file generated in the previous step is read back into a Spark DataFrame to verify successful storage.

In [ ]:
parquet_df = spark.read.parquet("input_parquet")

parquet_df.show(5)

+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
|user_id|product_id|   category|old_name|  price|base_price|amount|   status|region|priority|
+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
|   U001|      P101|Electronics|  Item_1| 178.18|       151|534.54|Completed| South|    High|
|   U002|      P102|Electronics|  Item_2| 1752.3|      1485|8761.5|Completed|  West|    High|
|   U003|      P103|Electronics|  Item_3| 343.38|       291|686.76|Completed| North|     Low|
|   U004|      P104|   Clothing|  Item_4|1847.88|      1566|9239.4|  Pending| South|  Medium|
|   U005|      P105|     Sports|  Item_5| 789.42|       669|789.42|Completed|  West|  Medium|
+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
only showing top 5 rows


## Step 11 — Handling Null Values

Rows containing null values in the **user_id** column are removed using the `isNotNull()` function.

In [ ]:
filtered_df = parquet_df.filter(
    col("user_id").isNotNull()
)

filtered_df.show(5)

+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
|user_id|product_id|   category|old_name|  price|base_price|amount|   status|region|priority|
+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
|   U001|      P101|Electronics|  Item_1| 178.18|       151|534.54|Completed| South|    High|
|   U002|      P102|Electronics|  Item_2| 1752.3|      1485|8761.5|Completed|  West|    High|
|   U003|      P103|Electronics|  Item_3| 343.38|       291|686.76|Completed| North|     Low|
|   U004|      P104|   Clothing|  Item_4|1847.88|      1566|9239.4|  Pending| South|  Medium|
|   U005|      P105|     Sports|  Item_5| 789.42|       669|789.42|Completed|  West|  Medium|
+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
only showing top 5 rows


## Step 12 — Writing Processed Data as CSV

The cleaned dataset is saved in CSV format using overwrite mode with column headers enabled.

In [ ]:
filtered_df.write \
    .mode("overwrite") \
    .option("header", True) \
    .csv("output_csv")

print("CSV Output Saved Successfully!")

CSV Output Saved Successfully!


## Step 13 — Verifying Generated Files

The generated Parquet and CSV output folders are inspected to verify successful execution of the Spark pipeline.

In [ ]:
import os

print("Files inside input_parquet:")
print(os.listdir("input_parquet"))

print("\nFiles inside output_csv:")
print(os.listdir("output_csv"))

Files inside input_parquet:
['._SUCCESS.crc', '.part-00000-15f4e03d-9d2f-493c-b5c8-625ebf5e5459-c000.snappy.parquet.crc', 'part-00000-15f4e03d-9d2f-493c-b5c8-625ebf5e5459-c000.snappy.parquet', '_SUCCESS']

Files inside output_csv:
['._SUCCESS.crc', 'part-00000-b3166943-8c39-4951-baad-21daa285a545-c000.csv', '_SUCCESS', '.part-00000-b3166943-8c39-4951-baad-21daa285a545-c000.csv.crc']


## Step 14 — Reading the Output CSV

The generated CSV file is loaded back into Spark to verify that the pipeline has successfully written the processed data.

In [ ]:
output_df = spark.read.csv(
    "output_csv",
    header=True,
    inferSchema=True
)

output_df.show(5)

+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
|user_id|product_id|   category|old_name|  price|base_price|amount|   status|region|priority|
+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
|   U001|      P101|Electronics|  Item_1| 178.18|       151|534.54|Completed| South|    High|
|   U002|      P102|Electronics|  Item_2| 1752.3|      1485|8761.5|Completed|  West|    High|
|   U003|      P103|Electronics|  Item_3| 343.38|       291|686.76|Completed| North|     Low|
|   U004|      P104|   Clothing|  Item_4|1847.88|      1566|9239.4|  Pending| South|  Medium|
|   U005|      P105|     Sports|  Item_5| 789.42|       669|789.42|Completed|  West|  Medium|
+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
only showing top 5 rows


## Step 15 — Filtering Using OR Conditions

The DataFrame is filtered to retrieve records where:

- Region is **North**
- OR Priority is **High**

In [ ]:
north_or_high = df.filter(
    (col("region") == "North") |
    (col("priority") == "High")
)

north_or_high.show()

+-------+----------+-----------+--------+-------+----------+-------+---------+------+--------+
|user_id|product_id|   category|old_name|  price|base_price| amount|   status|region|priority|
+-------+----------+-----------+--------+-------+----------+-------+---------+------+--------+
|   U001|      P101|Electronics|  Item_1| 178.18|       151| 534.54|Completed| South|    High|
|   U002|      P102|Electronics|  Item_2| 1752.3|      1485| 8761.5|Completed|  West|    High|
|   U003|      P103|Electronics|  Item_3| 343.38|       291| 686.76|Completed| North|     Low|
|   U006|      P106|    Grocery|  Item_6| 493.24|       418| 986.48|  Pending| North|    High|
|   U007|      P107|  Furniture|  Item_7| 351.64|       298|1054.92|  Pending|  East|    High|
|   U008|      P108|  Furniture|  Item_8|1413.64|      1198|1413.64|  Pending| North|     Low|
|   NULL|      P110|Electronics| Item_10| 227.74|       193| 455.48|  Pending| North|    High|
|   U011|      P111|Electronics| Item_11|1036.04| 

## Step 16 — Demonstrating Lazy Evaluation

Spark transformations are evaluated lazily. No computation is performed until an action such as `show()` or `count()` is executed.

In [ ]:
lazy_df = df.filter(
    col("amount") > 1000
).select(
    "user_id",
    "product_id",
    "amount"
)

print("Transformation created successfully.")
print("No execution has happened yet!")

Transformation created successfully.
No execution has happened yet!


## Triggering Execution

The `show()` action triggers the execution of all previously defined transformations.

In [ ]:
lazy_df.show()

+-------+----------+-------+
|user_id|product_id| amount|
+-------+----------+-------+
|   U002|      P102| 8761.5|
|   U004|      P104| 9239.4|
|   U007|      P107|1054.92|
|   U008|      P108|1413.64|
|   U009|      P109|10608.2|
|   U011|      P111|3108.12|
|   U012|      P112|1951.72|
|   U013|      P113|3178.92|
|   U014|      P114|3104.58|
|   U015|      P115|2154.68|
|   U017|      P117| 4295.2|
|   U018|      P118| 9587.5|
|   U019|      P119| 5764.3|
|   U020|      P120|5394.96|
|   U021|      P121| 3268.6|
|   U022|      P122|4191.36|
|   U023|      P123|1454.94|
|   U024|      P124|5794.98|
|   NULL|      P125| 2336.4|
|   U026|      P126| 2655.0|
+-------+----------+-------+
only showing top 20 rows


## Step 17 — Viewing the Execution Plan (DAG)

Spark generates an optimized execution plan before running a job. The `explain(True)` method displays the parsed, analyzed, optimized, and physical execution plans.

In [ ]:
lazy_df.explain(True)

== Parsed Logical Plan ==
'Project ['user_id, 'product_id, 'amount]
+- Filter (amount#23 > cast(1000 as double))
   +- Relation [user_id#17,product_id#18,category#19,old_name#20,price#21,base_price#22,amount#23,status#24,region#25,priority#26] csv

== Analyzed Logical Plan ==
user_id: string, product_id: string, amount: double
Project [user_id#17, product_id#18, amount#23]
+- Filter (amount#23 > cast(1000 as double))
   +- Relation [user_id#17,product_id#18,category#19,old_name#20,price#21,base_price#22,amount#23,status#24,region#25,priority#26] csv

== Optimized Logical Plan ==
Project [user_id#17, product_id#18, amount#23]
+- Filter (isnotnull(amount#23) AND (amount#23 > 1000.0))
   +- Relation [user_id#17,product_id#18,category#19,old_name#20,price#21,base_price#22,amount#23,status#24,region#25,priority#26] csv

== Physical Plan ==
*(1) Filter (isnotnull(amount#23) AND (amount#23 > 1000.0))
+- FileScan csv [user_id#17,product_id#18,amount#23] Batched: false, DataFilters: [isnotnull(

## Step 18 — Transformations vs Actions

Transformations create a new DataFrame without immediate execution.

Actions trigger the execution of the entire Spark job and produce a result.

In [ ]:
print("Transformation Example:")
transformation_df = df.select("user_id", "product_id")

print("Action Example:")
transformation_df.show(5)

Transformation Example:
Action Example:
+-------+----------+
|user_id|product_id|
+-------+----------+
|   U001|      P101|
|   U002|      P102|
|   U003|      P103|
|   U004|      P104|
|   U005|      P105|
+-------+----------+
only showing top 5 rows


## Step 19 — show() vs collect()

When exploring large datasets, `show()` is preferred because it retrieves only a small number of records.

Using `collect()` on very large datasets can cause excessive memory usage because it transfers all data to the Driver.

In [ ]:
print("Using show():")
df.show(5)

print("\nUsing collect() on only 5 rows:")
small_data = df.limit(5).collect()

for row in small_data:
    print(row)

Using show():
+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
|user_id|product_id|   category|old_name|  price|base_price|amount|   status|region|priority|
+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
|   U001|      P101|Electronics|  Item_1| 178.18|       151|534.54|Completed| South|    High|
|   U002|      P102|Electronics|  Item_2| 1752.3|      1485|8761.5|Completed|  West|    High|
|   U003|      P103|Electronics|  Item_3| 343.38|       291|686.76|Completed| North|     Low|
|   U004|      P104|   Clothing|  Item_4|1847.88|      1566|9239.4|  Pending| South|  Medium|
|   U005|      P105|     Sports|  Item_5| 789.42|       669|789.42|Completed|  West|  Medium|
+-------+----------+-----------+--------+-------+----------+------+---------+------+--------+
only showing top 5 rows

Using collect() on only 5 rows:
Row(user_id='U001', product_id='P101', category='Electronics', old_name='Item_1', p

# Conclusion

In this notebook, Apache Spark was used to perform end-to-end data processing using PySpark.

The assignment demonstrated:

- Reading CSV and Parquet files
- Schema handling
- DataFrame transformations
- Filtering and selection
- Renaming and casting columns
- Adding calculated columns
- Handling null values
- Writing processed data
- Lazy Evaluation
- DAG execution planning
- Spark best practices such as using `show()` instead of `collect()`

This practical implementation demonstrates how Spark efficiently processes large datasets while optimizing execution through lazy evaluation and columnar storage formats like Parquet.